In [1]:
#%pip install "https://github.com/RenatoVassallo/MacroPy/releases/download/0.1.5/macropy-0.1.5-py3-none-any.whl"
#%pip install -r "C:/Users/mlxfl/Documents/umich/699/project/requirements.txt"

In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import json
from io import StringIO
import statsmodels.api as sm
from statsmodels.stats.stattools import omni_normtest
from MacroPy import BayesianVAR

In [3]:
# Preset alpha value for the Bernferroni correction
alpha = 0.05

In [4]:
parent_dir = Path.cwd().parent.parent

In [5]:
# Import datasets (google trends will not be used for the regression analysis)

# Monthly
fin_gig_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'yfinance', 'gig_data_monthly.csv')) 
fin_sp500_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'yfinance', 'sp500_data_monthly.csv'))
fred_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'fred', 'fred_monthly_data.csv'))
#google_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'google_trends', 'google_trends_monthly.csv'))

# Quarter
fin_gig_q_df = pd.read_csv(os.path.join(parent_dir, 'data', 'yfinance', 'gig_data_quarterly.csv'))
fin_sp500_q_df = pd.read_csv(os.path.join(parent_dir, 'data', 'yfinance', 'sp500_data_quarterly.csv'))
fred_q_df = pd.read_csv(os.path.join(parent_dir, 'data', 'fred', 'fred_quarterly_data.csv'))
#google_q_df = pd.read_csv(os.path.join(parent_dir, 'data', 'google_trends', 'google_trends_quarterly.csv'))

# Annual
fin_gig_a_df = pd.read_csv(os.path.join(parent_dir, 'data', 'yfinance', 'gig_data_annually.csv'))
fin_sp500_a_df = pd.read_csv(os.path.join(parent_dir, 'data', 'yfinance', 'sp500_data_annually.csv'))
fred_a_df = pd.read_csv(os.path.join(parent_dir, 'data', 'fred', 'fred_annual_data.csv'))
#google_a_df = pd.read_csv(os.path.join(parent_dir, 'data', 'google_trends', 'google_trends_annual.csv'))

In [6]:
# # Get lag and non-lag datasets

# # Monthly
# # Non-lag
# fin_gig_m_nolag_df = fin_gig_m_df[[c for c in fin_gig_m_df.columns if 'lag' not in c]]
# fin_sp500_m_nolag_df = fin_sp500_m_df[[c for c in fin_sp500_m_df.columns if 'lag' not in c]]
# fred_m_nolag_df = fred_m_df[[c for c in fred_m_df.columns if 'lag' not in c and 'quarterly' not in c and 'annual' not in c]]
# #google_m_nolag_df = google_m_df[[c for c in google_m_df.columns if 'lag' not in c]]
# # Lag
# fin_gig_m_lag_df = fin_gig_m_df[[c for c in fin_gig_m_df.columns if 'lag' in c or c in ('Date','Month','Quarter','Year','Company','Ticker','industryKey','sectorKey')]]
# fin_sp500_m_lag_df = fin_sp500_m_df[[c for c in fin_sp500_m_df.columns if 'lag' in c or c in ('Date','Month','Quarter','Year','Company','Ticker','industryKey','sectorKey')]]
# fred_m_lag_df = fred_m_df[[c for c in fred_m_df.columns if ('lag' in c and 'quarterly' not in c and 'annual' not in c) or c in ('date','month','quarter','year')]]
# #google_m_lag_df = google_m_df[[c for c in google_m_df.columns if 'lag' in c or c in ('year','month','date')]]

# # Quarter
# # Non-lag
# fin_gig_q_nolag_df = fin_gig_q_df[[c for c in fin_gig_q_df.columns if 'lag' not in c]]
# fin_sp500_q_nolag_df = fin_sp500_q_df[[c for c in fin_sp500_q_df.columns if 'lag' not in c]]
# fred_q_nolag_df = fred_q_df[[c for c in fred_q_df.columns if 'lag' not in c and 'annual' not in c]]
# #google_q_nolag_df = google_q_df[[c for c in google_q_df.columns if 'lag' not in c]]
# # Lag
# fin_gig_q_lag_df = fin_gig_q_df[[c for c in fin_gig_q_df.columns if 'lag' in c or c in ('Date','Quarter','Year','Company','Ticker','industryKey','sectorKey')]]
# fin_sp500_q_lag_df = fin_sp500_q_df[[c for c in fin_sp500_q_df.columns if 'lag' in c or c in ('Date','Quarter','Year','Company','Ticker','industryKey','sectorKey')]]
# fred_q_lag_df = fred_q_df[[c for c in fred_q_df.columns if ('lag' in c and 'annual' not in c) or c in ('date','quarter','year')]]
# #google_q_lag_df = google_q_df[[c for c in google_q_df.columns if 'lag' in c or c in ('year','quarter','date')]]

# # Annual
# # Non-lag
# fin_gig_a_nolag_df = fin_gig_a_df[[c for c in fin_gig_a_df.columns if 'lag' not in c]]
# fin_sp500_a_nolag_df = fin_sp500_a_df[[c for c in fin_sp500_a_df.columns if 'lag' not in c]]
# fred_a_nolag_df = fred_a_df[[c for c in fred_a_df.columns if 'lag' not in c and 'quarterly' not in c]]
# #google_a_nolag_df = google_a_df[[c for c in google_a_df.columns if 'lag' not in c]]
# # Lag
# fin_gig_a_lag_df = fin_gig_a_df[[c for c in fin_gig_a_df.columns if 'lag' in c or c in ('Date','Year','Company','Ticker','industryKey','sectorKey')]]
# fin_sp500_a_lag_df = fin_sp500_a_df[[c for c in fin_sp500_a_df.columns if 'lag' in c or c in ('Date','Year','Company','Ticker','industryKey','sectorKey')]]
# fred_a_lag_df = fred_a_df[[c for c in fred_a_df.columns if 'lag' in c or c in ('date','year')]]
# #google_a_lag_df = google_a_df[[c for c in google_a_df.columns if 'lag' in c or c in ('year','date')]]

In [7]:
# # Merge datasets
# # Merging with the fred datasets being the driver because they go back the farthest
# # Note: statsmodels does not process nulls. So, for simplicity, they are going to be dropped

# # Monthly
# # Non-lag
# monthly_nolag_gig_df = fred_m_nolag_df.merge(fin_gig_m_nolag_df, left_on=['month','year'], right_on=['Month','Year'], how='left').dropna()#.merge(google_m_nolag_df, on=['month','year'], how='left')
# monthly_nolag_sp500_df = fred_m_nolag_df.merge(fin_sp500_m_nolag_df, left_on=['month','year'], right_on=['Month','Year'], how='left').dropna()#.merge(google_m_nolag_df, on=['month','year'], how='left')
# # Lag
# monthly_lag_gig_df = fred_m_lag_df.merge(fin_gig_m_lag_df, left_on=['month','year'], right_on=['Month','Year'], how='left').dropna()#.merge(google_m_lag_df, on=['month','year'], how='left')
# monthly_lag_sp500_df = fred_m_lag_df.merge(fin_sp500_m_lag_df, left_on=['month','year'], right_on=['Month','Year'], how='left').dropna()#.merge(google_m_lag_df, on=['month','year'], how='left')

# # Quarterly
# # Non-lag
# quarterly_nolag_gig_df = fred_q_nolag_df.merge(fin_gig_q_nolag_df, left_on=['quarter','year'], right_on=['Quarter','Year'], how='left').dropna()#.merge(google_q_nolag_df, on=['quarter','year'], how='left')
# quarterly_nolag_sp500_df = fred_q_nolag_df.merge(fin_sp500_q_nolag_df, left_on=['quarter','year'], right_on=['Quarter','Year'], how='left').dropna()#.merge(google_q_nolag_df, on=['quarter','year'], how='left')
# # Lag
# quarterly_lag_gig_df = fred_q_lag_df.merge(fin_gig_q_lag_df, left_on=['quarter','year'], right_on=['Quarter','Year'], how='left').dropna()#.merge(google_q_lag_df, on=['quarter','year'], how='left')
# quarterly_lag_sp500_df = fred_q_lag_df.merge(fin_sp500_q_lag_df, left_on=['quarter','year'], right_on=['Quarter','Year'], how='left').dropna()#.merge(google_q_lag_df, on=['quarter','year'], how='left')

# # Annual
# # Non-lag
# annual_nolag_gig_df = fred_a_nolag_df.merge(fin_gig_a_nolag_df, left_on=['year'], right_on=['Year'], how='left').dropna()#.merge(google_a_nolag_df, on=['year'], how='left')
# annual_nolag_sp500_df = fred_a_nolag_df.merge(fin_sp500_a_nolag_df, left_on=['year'], right_on=['Year'], how='left').dropna()#.merge(google_a_nolag_df, on=['year'], how='left')
# # Lag
# annual_lag_gig_df = fred_a_lag_df.merge(fin_gig_a_lag_df, left_on=['year'], right_on=['Year'], how='left').dropna()#.merge(google_a_lag_df, on=['year'], how='left')
# annual_lag_sp500_df = fred_a_lag_df.merge(fin_sp500_a_lag_df, left_on=['year'], right_on=['Year'], how='left').dropna()#.merge(google_a_lag_df, on=['year'], how='left')


In [8]:
# Merge datasets
# Merging with the fred datasets being the driver because they go back the farthest
# Note: statsmodels does not process nulls. So, for simplicity, they are going to be dropped

# Monthly
monthly_gig_df = fred_m_df.merge(fin_gig_m_df, left_on=['month','year'], right_on=['Month','Year'], how='left').dropna()#.merge(google_m_df, on=['month','year'], how='left')
monthly_sp500_df = fred_m_df.merge(fin_sp500_m_df, left_on=['month','year'], right_on=['Month','Year'], how='left').dropna()#.merge(google_m_df, on=['month','year'], how='left')

# Quarterly
quarterly_gig_df = fred_q_df.merge(fin_gig_q_df, left_on=['quarter','year'], right_on=['Quarter','Year'], how='left').dropna()#.merge(google_q_df, on=['quarter','year'], how='left')
quarterly_sp500_df = fred_q_df.merge(fin_sp500_q_df, left_on=['quarter','year'], right_on=['Quarter','Year'], how='left').dropna()#.merge(google_q_df, on=['quarter','year'], how='left')

# Annual
annual_gig_df = fred_a_df.merge(fin_gig_a_df, left_on=['year'], right_on=['Year'], how='left').dropna()#.merge(google_a_df, on=['year'], how='left')
annual_sp500_df = fred_a_df.merge(fin_sp500_a_df, left_on=['year'], right_on=['Year'], how='left').dropna()#.merge(google_a_df, on=['year'], how='left')


In [9]:
# # Regression analysis section

# # Get ys
# y_nolag_gig_monthly = monthly_nolag_gig_df['Close']
# y_nolag_sp500_monthly = monthly_nolag_sp500_df['Close']
# y_lag_gig_monthly = monthly_lag_gig_df['Close_lag1']
# y_lag_sp500_monthly = monthly_lag_sp500_df['Close_lag1']
# y_nolag_gig_quarterly = quarterly_nolag_gig_df['Close']
# y_nolag_sp500_quarterly = quarterly_nolag_sp500_df['Close']
# y_lag_gig_quarterly = quarterly_lag_gig_df['Close_lag1']
# y_lag_sp500_quarterly = quarterly_lag_sp500_df['Close_lag1']
# y_nolag_gig_annual = annual_nolag_gig_df['Close']
# y_nolag_sp500_annual = annual_nolag_sp500_df['Close']
# y_lag_gig_annual = annual_lag_gig_df['Close_lag1']
# y_lag_sp500_annual = annual_lag_sp500_df['Close_lag1']

In [10]:
# Regression analysis section

# Get ys
y_gig_monthly = monthly_gig_df['Close']
y_sp500_monthly = monthly_sp500_df['Close']
y_gig_quarterly = quarterly_gig_df['Close']
y_sp500_quarterly = quarterly_sp500_df['Close']
y_gig_annual = annual_gig_df['Close']
y_sp500_annual = annual_sp500_df['Close']

In [11]:
# # Regression analysis on quarterly data

# # Since Megan is doing correlation analysis on one variable, I will only do regression on at least 2 variables
# # Get all variable combinations for each dataset
# # I am just going to use the gig datasets arbitrarily for everything because the columns line up in both the gig and sp500 datasets
# # I will convert the dataset to a frozenset in order to get the unique column combinations, but then convert back to a list for processing

# # Monthly
# # No-lag columns 
# monthly_nolag_cols = []
# for c in monthly_nolag_gig_df.columns:
#     temp_cols = []
#     if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close']:
#         temp_cols.append(c)
#         for c2 in monthly_nolag_gig_df.columns:
#             if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close'] and c2 != c:
#                 temp_cols.append(c2)
#                 temp_col_set = frozenset(sorted(temp_cols))
#                 monthly_nolag_cols.append(temp_col_set)
# monthly_nolag_cols = frozenset(monthly_nolag_cols)
# monthly_nolag_cols = [sorted(i) for i in monthly_nolag_cols]
# # Lag columns
# monthly_lag_cols = []
# for c in monthly_lag_gig_df.columns:
#     temp_cols = []
#     if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter','Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close_lag1',
#        'Volume_lag1']:
#         temp_cols.append(c)
#         for c2 in monthly_lag_gig_df.columns:
#             if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close_lag1',
#        'Volume_lag1'] and c2 != c:
#                 temp_cols.append(c2)
#                 temp_col_set = frozenset(sorted(temp_cols))
#                 monthly_lag_cols.append(temp_col_set)
# monthly_lag_cols = frozenset(monthly_lag_cols)
# monthly_lag_cols = [sorted(i) for i in monthly_lag_cols]

# # Quarterly
# # No-lag columns
# quarterly_nolag_cols = []
# for c in quarterly_nolag_gig_df.columns:
#     temp_cols = []
#     if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close']:
#         temp_cols.append(c)
#         for c2 in quarterly_nolag_gig_df.columns:
#             if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close'] and c2 != c:
#                 temp_cols.append(c2)
#                 temp_col_set = frozenset(sorted(temp_cols))
#                 quarterly_nolag_cols.append(temp_col_set)
# quarterly_nolag_cols = frozenset(quarterly_nolag_cols)
# quarterly_nolag_cols = [sorted(i) for i in quarterly_nolag_cols]
# # Lag columns
# quarterly_lag_cols = []
# for c in quarterly_lag_gig_df.columns:
#     temp_cols = []
#     if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter','Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close_lag1',
#        'Volume_lag1']:
#         temp_cols.append(c)
#         for c2 in quarterly_lag_gig_df.columns:
#             if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close_lag1',
#        'Volume_lag1'] and c2 != c:
#                 temp_cols.append(c2)
#                 temp_col_set = frozenset(sorted(temp_cols))
#                 quarterly_lag_cols.append(temp_col_set)
# quarterly_lag_cols = frozenset(quarterly_lag_cols)
# quarterly_lag_cols = [sorted(i) for i in quarterly_lag_cols]

# # Annual 
# # No-lag columns
# annual_nolag_cols = []
# for c in annual_nolag_gig_df.columns:
#     temp_cols = []
#     if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close']:
#         temp_cols.append(c)
#         for c2 in annual_nolag_gig_df.columns:
#             if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close'] and c2 != c:
#                 temp_cols.append(c2)
#                 temp_col_set = frozenset(sorted(temp_cols))
#                 annual_nolag_cols.append(temp_col_set)
# annual_nolag_cols = frozenset(annual_nolag_cols)
# annual_nolag_cols = [sorted(i) for i in annual_nolag_cols]
# # Lag columns
# annual_lag_cols = []
# for c in annual_lag_gig_df.columns:
#     temp_cols = []
#     if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter','Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close_lag1',
#        'Volume_lag1']:
#         temp_cols.append(c)
#         for c2 in annual_lag_gig_df.columns:
#             if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close_lag1',
#        'Volume_lag1'] and c2 != c:
#                 temp_cols.append(c2)
#                 temp_col_set = frozenset(sorted(temp_cols))
#                 annual_lag_cols.append(temp_col_set)
# annual_lag_cols = frozenset(annual_lag_cols)
# annual_lag_cols = [sorted(i) for i in annual_lag_cols]

In [12]:
# Regression analysis on quarterly data

# Since Megan is doing correlation analysis on one variable, I will only do regression on at least 2 variables
# Get all variable combinations for each dataset
# I am just going to use the gig datasets arbitrarily for everything because the columns line up in both the gig and sp500 datasets
# I will convert the dataset to a frozenset in order to get the unique column combinations, but then convert back to a list for processing

# Monthly
monthly_cols = []
for c in monthly_gig_df.columns:
    temp_cols = []
    if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
       'sectorKey', 'Close']:
        temp_cols.append(c)
        for c2 in monthly_gig_df.columns:
            if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
       'sectorKey', 'Close'] and c2 != c:
                temp_cols.append(c2)
                temp_col_set = frozenset(sorted(temp_cols))
                monthly_cols.append(temp_col_set)
monthly_cols = frozenset(monthly_cols)
monthly_cols = [sorted(i) for i in monthly_cols]

# Quarterly
quarterly_cols = []
for c in quarterly_gig_df.columns:
    temp_cols = []
    if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
       'sectorKey', 'Close']:
        temp_cols.append(c)
        for c2 in quarterly_gig_df.columns:
            if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
       'sectorKey', 'Close'] and c2 != c:
                temp_cols.append(c2)
                temp_col_set = frozenset(sorted(temp_cols))
                quarterly_cols.append(temp_col_set)
quarterly_cols = frozenset(quarterly_cols)
quarterly_cols = [sorted(i) for i in quarterly_cols]

# Annual 
annual_cols = []
for c in annual_gig_df.columns:
    temp_cols = []
    if c not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
       'sectorKey', 'Close']:
        temp_cols.append(c)
        for c2 in annual_gig_df.columns:
            if c2 not in ['date', 'month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
       'sectorKey', 'Close'] and c2 != c:
                temp_cols.append(c2)
                temp_col_set = frozenset(sorted(temp_cols))
                annual_cols.append(temp_col_set)
annual_cols = frozenset(annual_cols)
annual_cols = [sorted(i) for i in annual_cols]

In [13]:
# # Bernferroni correction variable for each data frequency
# bonferroni_monthly_nolag = alpha / len(monthly_nolag_cols)
# bonferroni_monthly_lag = alpha / len(monthly_lag_cols)
# bonferroni_quarterly_nolag = alpha / len(quarterly_nolag_cols)
# bonferroni_quarterly_lag = alpha / len(quarterly_lag_cols)
# bonferroni_annual_nolag = alpha / len(annual_nolag_cols)
# bonferroni_annual_lag = alpha / len(annual_lag_cols)

In [14]:
# Bernferroni correction variable for each data frequency
bonferroni_monthly = alpha / len(monthly_cols)
bonferroni_quarterly = alpha / len(quarterly_cols)
bonferroni_annual = alpha / len(annual_cols)

In [15]:
# # Run regression on all column combinations and extract 
# #for x_cols in monthly_nolag_cols:
# model = sm.OLS(y_nolag_gig_monthly, monthly_nolag_gig_df[monthly_nolag_cols[0]]).fit()

# model_results ={
#    "columns": str(monthly_nolag_cols[0]),
#    "model_summary": str({"coef": model.params.to_dict(), "ci": model.conf_int().to_dict(), "p_values": model.pvalues.to_dict(), "t_values": model.tvalues.to_dict()}),
#    "r2": [float(model.rsquared)],
#    "r2_adj": [float(model.rsquared_adj)],
#    "f_score": [float(model.fvalue)],
#    "f_prob": [float(model.f_pvalue)],
#    "log_liklihood": [float(model.llf)],
#    "aic": [float(model.aic)],
#    "bic": [float(model.bic)],
#    "rse": [float(model.resid.std(ddof=len(monthly_nolag_cols[0])+1))] # 2nd answer from https://stackoverflow.com/questions/63333999/residual-standard-error-of-a-regression-in-python
# }

# for row in model.summary().tables[2].data:
#     for idx, val in enumerate(row):
#         if "Omnibus" in val:
#             model_results["omnibus"] = [float(row[idx+1])]
#         elif "Durbin-Watson" in val:
#             model_results["durbin_watson"] = [float(row[idx+1])]
#         elif "Prob(Omnibus)" in val:
#             model_results["omnibus_prob"] = [float(row[idx+1])]
#         elif "Jarque-Bera" in val:
#             model_results["jarque_bera"] = [float(row[idx+1])]
#         elif "Prob(Jarque-Bera)" in val:
#             model_results["jarque_bera_prob"] = [float(row[idx+1])]
#         elif "Skew" in val:
#             model_results["skew"] = [float(row[idx+1])]
#         elif "Kurtosis" in val:
#             model_results["kurtosis"] = [float(row[idx+1])]
#         elif "Cond." in val:
#             model_results["cond_no"] = [float(row[idx+1])]

# #model_summary = str(model_results.pop("model_summary"))
# #model_columns = model_results.pop("columns")
# model_df = pd.DataFrame.from_dict(model_results, orient='columns')
# #model_df["model_summary"] = model_summary

# model_df


In [21]:
# Bayesian VAR Analysis

# Monthly

monthly_gig_df_dateindex = monthly_gig_df.copy()#[monthly_cols]
monthly_cols = [c for c in monthly_gig_df.columns if '_lag' not in c and c not in ('month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
        'sectorKey')]
monthly_gig_df_dateindex = monthly_gig_df_dateindex[monthly_cols]
monthly_gig_df_dateindex['date'] = pd.to_datetime(monthly_gig_df_dateindex['date'])
monthly_gig_df_dateindex.set_index('date', inplace=True)

# y = pd.DataFrame(monthly_gig_df_dateindex['Close'])

# monthly_cols = [c for c in monthly_gig_df.columns if '_lag' in c or c in ('month', 'quarter', 'year','Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey',
#        'sectorKey', 'Close')]
# x = monthly_gig_df_dateindex.drop(columns=monthly_cols)

N = monthly_gig_df_dateindex.shape[1]
b_exo = np.zeros((N, N), dtype=int)

# make all other equations (i != Close_index) not depend on other variables
close_idx = list(monthly_gig_df_dateindex.columns).index('Close')
for i in range(N):
    if i == close_idx:
        continue
    for j in range(N):
        if j != i:
            b_exo[i, j] = 1

# Pass the data as a NumPy array and pass the restriction matrix (not b_exo==b_exo)
bvar_monthly_gig = BayesianVAR(monthly_gig_df_dateindex.to_numpy(), b_exo=b_exo, lags=4)
bvar_monthly_gig.model_summary()

ValueError: Input data 'y' must be a pandas DataFrame with a datetime index.

In [ ]:
#post_draws = bvar_monthly_gig.sample_posterior()
#post_irfs = bvar_monthly_gig.compute_irfs()
#print(post_draws)

Sampling Posterior:   0%|          | 0/5000 [02:31<?, ?it/s]


KeyboardInterrupt: 